# Small SfM probe: the registered count, the GPS prior covariance and the quiet logs

**What and why:** the orthomosaic `01a_sfm_orthomosaic` notebook takes about 2 hours, and the recent `rasterx.sfm` fixes
change only what it prints. This probe runs `sfm.run_sfm` on ONE GPS cluster of about 15 images of the staged Old Orchard
set (the dataset and image path of `orthomosaic/config_nb`, the `exif_gbx` read and the `cluster_by_gps` split of 01a) on
the same runtime (Serverless environment 5, env5 wheel), and checks:

- the registered count `run_sfm` returns and prints equals the images of the persisted model
  (`len(pycolmap.Reconstruction(...).images)`, summed over the models): the initial pair is counted;
- the first `[Mapping] k/N` progress line reads `3/N` (the initial pair, then the first added image);
- no `No pose priors with valid covariance found.` line: every GPS pose prior carries a covariance;
- no `could not set ...maxRecordsPerBatch` line and no `spark.conf.set` call from geobrix code during `run_sfm` (every
  call is recorded with its caller: the Spark Connect client sets keys of its own, such as
  `spark.databricks.connect.environment`);
- no native INFO line (`I2026...`) in the captured stdout and stderr (Python and file-descriptor level);
- the extraction input is `repartition(N, "source")`: its plan has `hashpartitioning(source`.

All outputs are SCRATCH: the feature and match tables `geospatial_docs.gopro.probe_sfm_small_<date>_*`, the persist
directory `/Volumes/geospatial_docs/orthomosaic/data/_probe/sfm_small_<date>` and a local `/tmp` directory. The shipped
models and tables of the series are never read or written. The scratch is deleted at the end, pass or fail.

**Expected output:** the chosen cluster (id, image count), the captured `run_sfm` output, one JSON of the checks
(`dbutils.notebook.exit`), and `PASS`. Any failed check raises `AssertionError` naming it.

**Expected time:** about 5 to 12 min after the environment install (the `exif_gbx` read of the 171 images about 1 min,
extraction, matching and mapping of 15 images 3 to 9 min: matching took 6 min on a cold Serverless run).

**Monitoring:** one printed line per step; the job's time limit ends a stuck run; cancel from the Runs page. No retries.

**Serverless / JOB usage:** no `%pip` cells; the runner installs the env5 wheel and pycolmap
(`gbx:test:notebooks-serverless --profile oauth-fe --notebook examples/tests/probe_sfm_small.ipynb --env-version 5
--extras vizx --extra-deps pycolmap==4.2.1`).


In [0]:
import contextlib, io, json, os, re, shutil, tempfile, time
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import functions as F

from databricks.labs import gbx
from databricks.labs.gbx.rasterx import sfm
from databricks.labs.gbx.rasterx.imagery import cluster_by_gps

_ = gbx.register(spark)  # the exif_gbx reader

# The series' dataset (orthomosaic/config_nb): staged images on the output Volume.
CATALOG_NAME = "geospatial_docs"
SCHEMA_NAME = "gopro"
DATASET_NAME = "OldOrchard_2017-07-22"
IMG_PATH = f"/Volumes/{CATALOG_NAME}/orthomosaic/data/{SCHEMA_NAME}/images/{DATASET_NAME}"
SENSOR_WIDTH_MM = 6.17
MAX_PAIR_DIST_M = 30
TARGET_IMAGES = 15

# SCRATCH only: never the series' persisted models or tables.
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
PROBE_DIR = f"/Volumes/{CATALOG_NAME}/orthomosaic/data/_probe/sfm_small_{STAMP}"
LOCAL_DIR = f"/tmp/probe_sfm_small_{STAMP}"
FEATURE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.probe_sfm_small_{STAMP}_features"
MATCH_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.probe_sfm_small_{STAMP}_matches"
for _p in (PROBE_DIR, LOCAL_DIR):
    assert "_probe" in _p or _p.startswith("/tmp/"), _p
print("scratch:", PROBE_DIR, LOCAL_DIR, FEATURE_TABLE, MATCH_TABLE, sep="\n  ")


geobrix 0.6.0: one wheel per runtime (+env5 / +env6); SQL functions register through databricks.labs.gbx.register(spark); the Python API is rasterx / gridx / vectorx. See https://databrickslabs.github.io/geobrix/docs/migration-0-6
geobrix: heavy (Catalyst) unavailable: Spark Connect session (Serverless); using light (48 functions)
geobrix: SQL unavailable: 5 functions (gbx_bng_cellarea, gbx_bng_eastnorthasbng, gbx_quadbin_distance, ...) need a SQL function wrapper, and SQL functions are unavailable here; using light
scratch:
  /Volumes/geospatial_docs/orthomosaic/data/_probe/sfm_small_20261112_143653
  /tmp/probe_sfm_small_20261112_143653
  geospatial_docs.gopro.probe_sfm_small_20261112_143653_features
  geospatial_docs.gopro.probe_sfm_small_20261112_143653_matches


In [0]:
# 01a's metadata read (exif_gbx; the QC filter of 01a kept all 171 images, so it is left out here),
# then its GPS clustering at a 15-image target; the cluster nearest 15 images is the probe's input.
_t0 = time.perf_counter()
from databricks.labs.gbx.ds.exif import build_exif_common_opts

_exif_opts, _ = build_exif_common_opts(IMG_PATH, mode="sample", limit=24)
_exif_opts["sensorWidthMm"] = str(SENSOR_WIDTH_MM)
df_meta = spark.read.format("exif_gbx").options(**_exif_opts).load(IMG_PATH)
df_all = (
    df_meta.withColumn("gps_geom", F.expr("st_point(longitude, latitude)"))
    .withColumnRenamed("path", "source")
)
_pdf = df_all.select("source", "latitude", "longitude").toPandas()
_clustered = cluster_by_gps(_pdf, target_cluster_images=TARGET_IMAGES, overlap_frac=0.30, min_cluster_images=8)
_sizes = _clustered.groupby("_cluster").size().to_dict()
CID = min(_sizes, key=lambda c: (abs(_sizes[c] - TARGET_IMAGES), c))
SRCS = sorted(_clustered.loc[_clustered["_cluster"] == CID, "source"].tolist())
df_cluster = df_all.filter(F.col("source").isin(SRCS))
N = df_cluster.count()
print(f"{len(_pdf)} images -> {len(_sizes)} GPS clusters at target {TARGET_IMAGES}; "
      f"probe cluster {CID}: {N} images ({time.perf_counter() - _t0:.0f}s)")
assert N == len(SRCS) and 10 <= N <= 25, (N, len(SRCS))


171 images -> 16 GPS clusters at target 15; probe cluster 0: 15 images (44s)


In [0]:
# The extraction input run_sfm builds (sfm._one_image_per_partition): repartition(N, "source").
_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    sfm._one_image_per_partition(df_cluster, N).explain(True)
PLAN = _buf.getvalue()
PLAN_HASH_SOURCE = bool(re.search(r"hashpartitioning\(source", PLAN))
print([ln.strip() for ln in PLAN.splitlines() if "hashpartitioning" in ln or "Repartition" in ln][:4])
print("extraction plan has hashpartitioning(source):", PLAN_HASH_SOURCE)


["RepartitionByExpression ['source], 15, 0, Unknown", 'RepartitionByExpression [source#10633], 15, 0, Unknown', 'RepartitionByExpression [source#10633], 15, 0, Unknown', '+- PhotonShuffleExchangeSink hashpartitioning(source#10633, 15)']
extraction plan has hashpartitioning(source): True


In [0]:
# run_sfm into SCRATCH, capturing its Python output and the process's fd 1 / fd 2 (native glog lines),
# with every spark.conf.set call recorded. Checks, then the scratch is deleted, pass or fail.
import pycolmap

import sys

CONF_SETS = []      # every spark.conf.set during run_sfm: (key, the calling file)
GBX_CONF_SETS = []  # those whose caller is geobrix code (the Spark Connect client sets its own keys)
try:
    from pyspark.sql.connect.conf import RuntimeConf as _RC
except Exception:  # classic session
    from pyspark.sql.conf import RuntimeConfig as _RC
_orig_set = _RC.set
def _rec_set(self, key, value):
    _caller = sys._getframe(1).f_code.co_filename
    CONF_SETS.append([str(key), _caller])
    if "/databricks/labs/gbx/" in _caller:
        GBX_CONF_SETS.append(str(key))
    return _orig_set(self, key, value)

checks = {}
try:
    _native = tempfile.NamedTemporaryFile(prefix="probe_sfm_fd_", suffix=".log", delete=False)
    _py = io.StringIO()
    _saved = (os.dup(1), os.dup(2))
    _t0 = time.perf_counter()
    _RC.set = _rec_set
    try:
        os.dup2(_native.fileno(), 1)
        os.dup2(_native.fileno(), 2)
        with contextlib.redirect_stdout(_py), contextlib.redirect_stderr(_py):
            res = sfm.run_sfm(
                spark,
                df_qc=df_cluster,
                image_dir=IMG_PATH,
                output_dir=LOCAL_DIR,
                group_key=f"probe_c{CID}",
                feature_table=FEATURE_TABLE,
                match_table=MATCH_TABLE,
                max_pair_dist_m=MAX_PAIR_DIST_M,
                force_features=True,
                force_matches=True,
                force_master=True,
                persist_dir=PROBE_DIR,
            )
    finally:
        _RC.set = _orig_set
        os.dup2(_saved[0], 1)
        os.dup2(_saved[1], 2)
        os.close(_saved[0])
        os.close(_saved[1])
        _native.close()
    SFM_SECS = time.perf_counter() - _t0
    PY_OUT = _py.getvalue()
    NATIVE_OUT = Path(_native.name).read_text(errors="replace")
    os.unlink(_native.name)
    print(PY_OUT)
    print(f"--- fd 1/2 capture: {len(NATIVE_OUT.splitlines())} lines ---")
    print("\n".join(NATIVE_OUT.splitlines()[:40]))
    ALL_OUT = PY_OUT + "\n" + NATIVE_OUT

    # The persisted model(s): every model directory under PROBE_DIR/sparse.
    _models = sorted(p.parent for p in Path(PROBE_DIR, "sparse").rglob("images.bin"))
    MODEL_IMAGES = {str(m.relative_to(PROBE_DIR)): len(pycolmap.Reconstruction(m).images) for m in _models}
    _m = re.search(r"\((\d+)/(\d+) images registered\)", PY_OUT)
    _first = re.search(r"\[Mapping\] (\d+)/(\d+) images", PY_OUT)
    checks = {
        "cluster": int(CID),
        "images": N,
        "run_sfm_secs": round(SFM_SECS, 1),
        "registered_returned": res.get("registered"),
        "registered_printed": int(_m.group(1)) if _m else None,
        "printed_of": int(_m.group(2)) if _m else None,
        "model_images": MODEL_IMAGES,
        "model_images_total": sum(MODEL_IMAGES.values()),
        "first_progress": _first.group(0) if _first else None,
        "no_covariance_warning": "No pose priors with valid covariance found" not in ALL_OUT,
        "no_could_not_set": "could not set" not in ALL_OUT,
        "conf_set_calls": CONF_SETS,
        "gbx_conf_set_calls": GBX_CONF_SETS,
        "native_info_lines": len(re.findall(r"(?m)^I2026", ALL_OUT)),
        "native_lines_total": len(NATIVE_OUT.splitlines()),
        "plan_hashpartitioning_source": PLAN_HASH_SOURCE,
    }
    print(json.dumps(checks, indent=2))
    assert MODEL_IMAGES, "no persisted model under the probe's persist dir"
    assert checks["registered_returned"] == checks["model_images_total"], "returned registered != model images"
    assert checks["registered_printed"] == checks["model_images_total"], "printed registered != model images"
    assert checks["first_progress"] is not None and int(_first.group(1)) == 3, "first progress line is not 3/N"
    assert checks["no_covariance_warning"], "the 'No pose priors with valid covariance' warning appeared"
    assert checks["no_could_not_set"], "a 'could not set' line appeared"
    assert not GBX_CONF_SETS, f"geobrix code called spark.conf.set: {GBX_CONF_SETS}"
    assert checks["native_info_lines"] == 0, "native INFO (I2026...) lines appeared"
    assert PLAN_HASH_SOURCE, "the extraction plan has no hashpartitioning(source)"
    checks["pass"] = True
    print("\nPASS: registered", checks["registered_returned"], "==", checks["model_images_total"],
          "model images; no covariance warning, no conf set, no native INFO; hash-partitioned extraction")
finally:
    for _t in (FEATURE_TABLE, MATCH_TABLE):
        spark.sql(f"DROP TABLE IF EXISTS {_t}")
    shutil.rmtree(PROBE_DIR, ignore_errors=True)
    shutil.rmtree(LOCAL_DIR, ignore_errors=True)
    _left = [_t for _t in (FEATURE_TABLE, MATCH_TABLE) if spark.catalog.tableExists(_t)]
    print("scratch removed:", not Path(PROBE_DIR).exists() and not _left, _left)
dbutils.notebook.exit(json.dumps(checks))  # surface the result to jobs.get_run_output
